# Sentence segmentation for `thinking_text`

Inserts `⟦S1⟧ ⟦S2⟧ …` markers at sentence starts, **without changing a single original character**.
Removing the markers (`⟦S<number>⟧`) always gives back the exact input.

Approach: pure-Python rules + contextual checks (no LLM, no downloads, only `pandas`).
Run the sections top to bottom; **inspect the preview (section 4) before saving (section 6)**.

## 1. Configuration

In [6]:
from pathlib import Path

INPUT_PATH  = "../../data/raw/LiquidAILFM2.5-2.6B/LiquidAILFM2.5-2.6B_raw_results.csv"            # .csv, .tsv or .parquet
# Where to save the result (.csv, .tsv or .parquet). Default: next to the input, same format, "_segmented" suffix.
OUTPUT_PATH = "../../data/processed/LiquidAILFM2.5-2.6B/LiquidAILFM2.5-2.6B_raw_results_segmented.csv" 
TEXT_COL     = "thinking_text"
CSV_KWARGS   = {}      # extra pandas.read_csv options, e.g. {"sep": "\t"} or {"keep_default_na": False}
SEED         = 42      # reproducible preview sampling
N_PREVIEW    = 10      # rows shown in section 4
MARKER_START = 1       # first marker is ⟦S1⟧

## 2. Load data

In [7]:
import re, html
from pathlib import Path
import pandas as pd
from IPython.display import HTML, display

def load_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {path.resolve()}")
    ext = path.suffix.lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in (".csv", ".tsv"):
        kwargs = dict(CSV_KWARGS)
        if ext == ".tsv":
            kwargs.setdefault("sep", "\t")
        return pd.read_csv(path, **kwargs)
    raise ValueError(f"Unsupported input format '{ext}' (use .csv, .tsv or .parquet)")

df = load_table(INPUT_PATH)

if TEXT_COL not in df.columns:
    raise KeyError(f"Column '{TEXT_COL}' not found. Available columns: {list(df.columns)}")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
n_missing = df[TEXT_COL].isna().sum()
n_blank = df[TEXT_COL].dropna().astype(str).str.strip().eq("").sum()
print(f"{TEXT_COL}: {n_missing} missing, {n_blank} empty/whitespace-only, "
      f"{len(df) - n_missing - n_blank} with text")
df.head(3)

Shape: (29050, 17)
Columns: ['question_id', 'benchmark', 'difficulty_tier', 'task_type', 'compute_level', 'sample_index', 'raw_model_output', 'thinking_text', 'gold', 'is_correct', 'tokens_used', 'thinking_tokens', 'output_tokens', 'hit_response_marker', 'latency_s', 'was_truncated', 'hit_marker_empty_output']
thinking_text: 4151 missing, 0 empty/whitespace-only, 24899 with text


,question_id,benchmark,difficulty_tier,task_type,compute_level,sample_index,raw_model_output,thinking_text,gold,is_correct,tokens_used,thinking_tokens,output_tokens,hit_response_marker,latency_s,was_truncated,hit_marker_empty_output
0,GSM8K_0,GSM8K,easy,math,0,2,Janet's ducks lay **16 eggs** per day. \nShe ...,NaN,18,True,96,0,93,True,2.883648,False,False
1,GSM8K_0,GSM8K,easy,math,0,1,Janet's ducks lay 16 eggs per day. She uses 3 ...,NaN,18,True,113,0,110,True,3.440018,False,False
2,GSM8K_0,GSM8K,easy,math,0,0,Janet's ducks lay 16 eggs each day. From these...,NaN,18,True,138,0,135,True,4.148850,False,False


## 3. Segmentation logic

Rules, in short:

* A sentence can end at `.` `!` `?` `…` (plus closing quotes/brackets/`*`) **only if whitespace follows** and the next token looks like a sentence start (capital letter, digit, opening quote/bracket, `$`, `\`, …). So `3.14`, `1.2e-5`, `file.py`, `x.y` never split, and lowercase continuations (`e.g. this`, `etc. and`) don't either.
* For a `.`, the word before it is checked: titles/reference abbreviations (`Mr.`, `Dr.`, `vs.`, `Fig.`, `e.g.`, …) never end a sentence; list numbers and option labels (`1.`, `A.`, `B.` after `A.`) are not ends; dotted acronyms (`U.S.`) and initials (`John F. Kennedy`) are ambiguous and only split when a typical sentence opener follows (`… in the U.S. The …`).
* Newlines are clues, not automatic boundaries: a blank line splits unless the sentence clearly continues (`,`/`;` before it, or lowercase after it); a single newline splits only before list/option/heading lines or after a tag-only line like `<think>`.
* Markers go *before* the first character of each sentence; surrounding whitespace is never touched.
* **A line ending in `:` never ends a segment.** Every line after it (equations, code, list items, even a short sentence) stays in the same segment until an ordinary prose line resumes (capitalised, 4+ mostly plain words, or a new intro line ending in `:`).
* **Very short fillers** (`Hmm.`, `Wait, no.`, `Okay, so.`, `Hold on.`; at most 3 words, all from a small interjection list `FILLER_WORDS`) are merged into the sentence that follows, even across a blank line. A short sentence with any real content word (`Done.`, `Next step.`) is not a filler.


In [8]:
MARKER_START = 1
MARKER_RE = re.compile(r"⟦S\d+⟧")

# ---- small, general-purpose word lists (not per-case exceptions) ----------------------------
# Abbreviations that essentially never end a sentence (titles, "compare", figure/equation refs...).
NEVER_END = {
    "mr", "mrs", "ms", "mx", "dr", "prof", "sr", "jr", "st", "mt", "gen", "col", "capt", "sgt",
    "lt", "rev", "hon", "vs", "cf", "viz", "fig", "figs", "eq", "eqs", "vol", "pp", "approx",
    "e.g", "i.e",
}
# Function words that typically open a sentence. Only used to settle *ambiguous* cases
# (initials, dotted acronyms like "U.S.", "Option D. Let me ...").
STARTERS = {
    "the", "a", "an", "this", "that", "these", "those", "it", "its", "he", "she", "they", "we",
    "you", "i", "let", "let's", "so", "then", "thus", "therefore", "hence", "however", "but",
    "and", "or", "now", "next", "first", "second", "finally", "also", "if", "in", "on", "at",
    "for", "as", "wait", "hmm", "okay", "ok", "yes", "no", "maybe", "perhaps", "actually",
    "alternatively", "similarly", "since", "because", "there", "here", "what", "why", "how",
    "when", "which", "who", "to", "from", "with", "note", "check", "given", "suppose",
    "assume", "consider", "looking", "but",
}

OPENERS = "\"'“‘([{*_`"                 # characters that may precede the first word of a sentence
CLOSERS = "\"'”’)\\]}*_`»"              # characters that may trail sentence-ending punctuation

# terminal punctuation run + optional closers + whitespace, followed by more text
_CAND = re.compile(r"(?P<p>[.!?…]+)(?P<c>[" + re.escape(CLOSERS) + r"]*)(?P<ws>\s+)(?=\S)")
# whitespace containing at least one newline between two non-space characters
_NL = re.compile(r"(?<=\S)(?P<ws>[ \t]*\r?\n\s*)(?=\S)")
# a line that begins a list item / option label / heading
_LIST_LINE = re.compile(r"(?:[-*•+]\s|\d{1,3}[.)]\s|[A-Za-z][.)]\s|#{1,6}\s|\([a-zA-Z0-9]{1,3}\)\s)")
_TAG_LINE = re.compile(r"\s*</?[\w-]+>\s*")
_DOTTED = re.compile(r"(?:[A-Za-z]\.)+[A-Za-z]")
_NEXT_WORD = re.compile(r"[\"'“‘(\[{*_`]*([A-Za-z][A-Za-z']*)")


def _starts_sentence(text, i):
    """Does text[i:] look like the beginning of a new sentence?"""
    n = len(text)
    while i < n and text[i] in OPENERS:
        i += 1
    if i >= n:
        return False
    c = text[i]
    return c.isupper() or c.isdigit() or c in "$\\#<"


def _next_word(text, i):
    m = _NEXT_WORD.match(text, i)
    return m.group(1).lower() if m else ""


def _at_line_start(text, j):
    k = j
    while k > 0 and text[k - 1] in " \t":
        k -= 1
    return k == 0 or text[k - 1] == "\n"


def _prev_nonspace(text, j):
    k = j
    while k > 0 and text[k - 1].isspace():
        k -= 1
    return text[k - 1] if k > 0 else ""


def _period_is_boundary(text, m):
    """Contextual checks for a single '.' followed by whitespace. Returns True if a sentence ends here."""
    ws_start = m.start("ws")
    ws_end = m.end("ws")
    has_newline = "\n" in m.group("ws")
    if not (_starts_sentence(text, ws_end) or (has_newline and text[ws_end:ws_end + 1].strip())):
        return False                                    # lowercase continuation: "e.g. this", "etc. and"

    # token immediately before the period (without leading quotes/brackets)
    p = m.start("p")
    j = p
    while j > 0 and not text[j - 1].isspace():
        j -= 1
    tok = text[j:p].lstrip(OPENERS).rstrip("*_`'\"")
    low = tok.lower()

    if low in NEVER_END:
        return False
    if not tok:
        return True

    prev_ch = _prev_nonspace(text, j)

    # list numbering / option labels: "1.", "12.", "A." at the start of a line or sentence
    is_label_pos = not m.group("c")                     # a label like "1." has no closing quote/bracket
    if is_label_pos and re.fullmatch(r"\d{1,3}", tok) and (_at_line_start(text, j) or prev_ch in ".!?:"):
        return False
    if is_label_pos and re.fullmatch(r"[A-Za-z]", tok) and (_at_line_start(text, j) or prev_ch in ".!?"):
        return False

    nxt = _next_word(text, ws_end)
    # dotted acronyms ("U.S.", "a.m.", "Ph.D.") are ambiguous -> need a sentence-starter next
    if _DOTTED.fullmatch(tok):
        return has_newline or nxt in STARTERS
    # single capital letter mid-sentence: option label, initial ("John F. Kennedy"), or end ("... go with D. Let me")
    if re.fullmatch(r"[A-Z]", tok):
        if tok != "A" and re.search(rf"(?<![A-Za-z]){chr(ord(tok) - 1)}\.\s", text[max(0, j - 200):j]):
            return False                                # next letter of a nearby "A. ... B. ... C." option list
        if prev_ch in ":,;(":                            # "choices: A. 10^-8 eV, B. ..." vs "answer: B. Then ..."
            return has_newline or nxt in STARTERS
        words = text[max(0, j - 80):j].split()
        prev_word = words[-1] if words else ""
        looks_like_name = prev_word[:1].isupper() and prev_word[-1:].isalpha()
        if looks_like_name and not has_newline:
            return nxt in STARTERS
        return True
    return True

# Interjections / hesitation words. A very short sentence made ONLY of these ("Hmm.", "Wait, no.",
# "Okay, so.", "Hold on.") is not a thought on its own: it is merged into the sentence that follows.
FILLER_WORDS = {
    "hmm", "hmmm", "hm", "um", "uh", "huh", "oh", "ah", "aha", "wait", "hold", "on", "no", "nope",
    "yes", "yeah", "yep", "okay", "ok", "alright", "right", "sure", "so", "well", "now", "but",
    "actually", "again", "what",
}
MAX_FILLER_WORDS = 3
_FILLER_SEG = re.compile(r"[\"'“‘(\[{*_`]*([A-Za-z][A-Za-z']*(?:[ ,]+[A-Za-z][A-Za-z']*){0,%d}),?[.!?…]+[\"'”’)\]}*_`]*" % (MAX_FILLER_WORDS - 1))
_WORD = re.compile(r"[A-Za-z][A-Za-z'’]*[,.;:!?]?")


def _is_filler(seg):
    """'Hmm.', 'Wait, no.', 'Okay, so.' -> True. Anything containing a non-filler word -> False."""
    m = _FILLER_SEG.fullmatch(seg.strip())
    if not m:
        return False
    words = re.findall(r"[A-Za-z']+", m.group(1).lower())
    return 0 < len(words) <= MAX_FILLER_WORDS and all(w in FILLER_WORDS for w in words)


def _is_prose_line(line):
    """Ordinary prose line (capitalised, mostly plain words). Decides where a ':' block ends."""
    c = line.strip()
    if _LIST_LINE.match(c) or line[:1] in " \t":
        return False
    first = next((ch for ch in c if ch.isalpha()), "")
    if not first.isupper():
        return False
    toks = c.split()
    if sum(1 for t in toks if _WORD.fullmatch(t)) / len(toks) < 0.6:
        return False
    return len(toks) >= 4 or (c.endswith(":") and len(toks) >= 3)


def colon_blocks(text):
    """Spans (a, b) where NO boundary may be inserted: everything after a line ending in ':'
    (equations, code, list items, short sentences ...) stays in the same segment until an
    ordinary prose line resumes."""
    spans, in_block, first_after, block_from, pos = [], False, False, 0, 0
    for line in text.splitlines(keepends=True):
        start, pos = pos, pos + len(line)
        content = line.strip()
        if not content:
            continue
        off = start + (len(line) - len(line.lstrip()))
        if in_block:
            joins = first_after or not _is_prose_line(line)
            first_after = False
            if joins:
                continue
            spans.append((block_from, off))
            in_block = False
        if content.endswith(":"):
            in_block, first_after, block_from = True, True, start + len(line.rstrip())
    if in_block:
        spans.append((block_from, len(text)))
    return spans


def boundary_starts(text):
    """Character offsets where a new sentence begins (excluding the very first one)."""
    starts = set()

    for m in _CAND.finditer(text):
        p = m.group("p")
        if p == ".":
            ok = _period_is_boundary(text, m)
        else:                                           # '!', '?', '...', '…'
            ok = _starts_sentence(text, m.end("ws")) or "\n" in m.group("ws")
        if ok:
            starts.add(m.end("ws"))

    for m in _NL.finditer(text):
        ws, a, b = m.group("ws"), m.start("ws"), m.end("ws")
        prev_ch = text[a - 1]
        nxt_ch = text[b]
        if ws.count("\n") >= 2:                         # paragraph break
            after_stop = text[max(0, a - 8):a].rstrip(CLOSERS)[-1:] in (".", "!", "?", ":", "…")
            if prev_ch in ",;" or (nxt_ch.islower() and not after_stop):
                continue                                # sentence clearly continues over the blank line
            starts.add(b)
        else:                                           # single newline: only with structural clues
            line = text[text.rfind("\n", 0, a) + 1:a]
            if _LIST_LINE.match(text, b) or _TAG_LINE.fullmatch(line):
                starts.add(b)
    return sorted(starts)


def segment_text(text, start=MARKER_START):
    """Insert ⟦S<n>⟧ markers at sentence starts. Removing the markers returns `text` exactly."""
    if not isinstance(text, str) or not text.strip():
        return text
    if MARKER_RE.search(text):
        raise ValueError("input already contains ⟦S#⟧ markers")
    first = len(text) - len(text.lstrip())
    blocks = colon_blocks(text)
    cand = [s for s in boundary_starts(text) if s > first and not any(a < s < b for a, b in blocks)]
    starts = [first] + cand
    ends = starts[1:] + [len(text)]
    kept, i = [], 0                                  # merge filler-only segments into the following one
    while i < len(starts):
        kept.append(starts[i])
        j = i
        while j + 1 < len(starts) and _is_filler(text[starts[j]:ends[j]]):
            j += 1
        i = j + 1
    out, prev = [], 0
    for k, s_ in enumerate(kept):
        out.append(text[prev:s_])
        out.append(f"⟦S{start + k}⟧")
        prev = s_
    out.append(text[prev:])
    return "".join(out)


def strip_markers(text):
    return MARKER_RE.sub("", text) if isinstance(text, str) else text

### Acceptance tests (exact text preservation + expected markers)

In [9]:
def check(text, expected):
    """expected = exact marked output; also verifies the original is recoverable."""
    out = segment_text(text)
    assert strip_markers(out) == text, f"TEXT CHANGED for {text!r}"
    assert out == expected, f"\n  input:    {text!r}\n  got:      {out!r}\n  expected: {expected!r}"

# required cases
check("Mr. X is here. He is ready.",                       "⟦S1⟧Mr. X is here. ⟦S2⟧He is ready.")
check("The result is 3.14. Therefore, it is valid.",       "⟦S1⟧The result is 3.14. ⟦S2⟧Therefore, it is valid.")
check("For example, x = 2.5 and y = 3.0.",                 "⟦S1⟧For example, x = 2.5 and y = 3.0.")
check("We use e.g. this approach. It works.",              "⟦S1⟧We use e.g. this approach. ⟦S2⟧It works.")
check("The value is approximately 1.2e-5.",                "⟦S1⟧The value is approximately 1.2e-5.")
check('He wrote "This is correct." Then he continued.',    '⟦S1⟧He wrote "This is correct." ⟦S2⟧Then he continued.')
check("The equation is x = 2. The next step is y = 3.",    "⟦S1⟧The equation is x = 2. ⟦S2⟧The next step is y = 3.")

# newlines, parentheses, math, lists
check("Let me think (carefully).\n\nFirst, x = 3.5 * (2 + 1) = 10.5.\nSo y = 10.5 - 0.5 = 10.0! Is that right?\n\n- Check: 10.0 > 9.\n- Done.",
      "⟦S1⟧Let me think (carefully).\n\n⟦S2⟧First, x = 3.5 * (2 + 1) = 10.5.\n⟦S3⟧So y = 10.5 - 0.5 = 10.0! ⟦S4⟧Is that right?\n\n⟦S5⟧- Check: 10.0 > 9.\n⟦S6⟧- Done.")
check("1. Check it. If yes, go.\n2. Otherwise stop.",       "⟦S1⟧1. Check it. ⟦S2⟧If yes, go.\n⟦S3⟧2. Otherwise stop.")
check("Options:\nA. Safe, B. Unsafe.\nWe pick D. Done.",    "⟦S1⟧Options:\nA. Safe, B. Unsafe.\n⟦S2⟧We pick D. ⟦S3⟧Done.")
check("<think>\nOkay, so. Let me see.",                    "⟦S1⟧<think>\n⟦S2⟧Okay, so. Let me see.")
check("In the U.S. The next step. John F. Kennedy spoke.", "⟦S1⟧In the U.S. ⟦S2⟧The next step. ⟦S3⟧John F. Kennedy spoke.")
check("Is it done? yes, see Fig. 3.) Then stop...  Next.", "⟦S1⟧Is it done? yes, see Fig. 3.) ⟦S2⟧Then stop...  ⟦S3⟧Next.")
check("  Leading space. Trailing space.  \n",              "  ⟦S1⟧Leading space. ⟦S2⟧Trailing space.  \n")

# short fillers merge into the next sentence (even across a blank line)
check("Hmm.\n\nWait, no. Wait, the operation is to swap two cards once, but only if it's needed.",
      "⟦S1⟧Hmm.\n\nWait, no. Wait, the operation is to swap two cards once, but only if it's needed.")
check("Hold on. Done.\n\nNext step.", "⟦S1⟧Hold on. Done.\n\n⟦S2⟧Next step.")   # "Done." has content -> not a filler
check("Hmm.", "⟦S1⟧Hmm.")                                                        # a trailing filler stays on its own

# every line after a line ending in ':' (code, equations, lists) stays in the same segment
check("Else:\n\nfor each pair of indices i, j (i < j) in 0, 1, 2:\n\n   swapped = s with s[i] and s[j] swapped.\n\nif swapped == \"abc\":\n\n       return YES for this test case.\n\nSo in code:\n\nRead t test cases.\n\nThen we loop over everything carefully.",
      "⟦S1⟧Else:\n\nfor each pair of indices i, j (i < j) in 0, 1, 2:\n\n   swapped = s with s[i] and s[j] swapped.\n\nif swapped == \"abc\":\n\n       return YES for this test case.\n\n⟦S2⟧So in code:\n\nRead t test cases.\n\n⟦S3⟧Then we loop over everything carefully.")
check("So, setting each factor equal to zero gives:\n\n\\( x + 3 = 0 \\) => \\( x = -3 \\)\n\n\\( x - 2 = 0 \\) => \\( x = 2 \\)\n\nTherefore, the solutions are x = -3 and x = 2.",
      "⟦S1⟧So, setting each factor equal to zero gives:\n\n\\( x + 3 = 0 \\) => \\( x = -3 \\)\n\n\\( x - 2 = 0 \\) => \\( x = 2 \\)\n\n⟦S2⟧Therefore, the solutions are x = -3 and x = 2.")

# empty / missing / non-text values come back untouched
for v in ["", "   ", "\n\n", None, float("nan"), 5]:
    out = segment_text(v)
    assert out is v or (v != v and out != out) or out == v, f"changed {v!r}"

# existing markers must be refused, not silently mangled
try:
    segment_text("Already ⟦S1⟧marked."); raise SystemExit("expected ValueError")
except ValueError:
    pass
print("All acceptance tests passed.")

All acceptance tests passed.


Apply to the DataFrame. Errors are raised with the offending row index (nothing is skipped silently); the original `df` is left untouched.

In [10]:
def segment_column(series):
    results = []
    for idx, value in series.items():
        try:
            results.append(segment_text(value, start=MARKER_START))
        except Exception as exc:
            raise RuntimeError(f"Segmentation failed at row index {idx!r}: {exc}") from exc
    return pd.Series(results, index=series.index, dtype=series.dtype, name=series.name)

if df[TEXT_COL].astype(str).str.contains(MARKER_RE).any():
    raise ValueError("Some input rows already contain ⟦S#⟧ markers; remove or rename them first.")

out = df.copy()
out[TEXT_COL] = segment_column(df[TEXT_COL])
print("Done:", len(out), "rows processed.")

Done: 29050 rows processed.


## 4. Preview and manual inspection

Original (left) next to segmented (right). Markers are highlighted; long texts scroll inside their box, nothing is truncated.
The sample mixes random rows with rows that contain edge cases (abbreviations, decimals, equations, code, lists, quotes) when available.

In [11]:
EDGE_PATTERNS = {
    "abbreviation": r"\b(?:Mr|Mrs|Ms|Dr|Prof|vs|etc|Fig)\.|\b(?:[A-Za-z]\.){2,}",
    "decimal":      r"\d\.\d",
    "equation":     r"\d\s*[-+*/×÷^=]\s*[-$\d]|\s=\s",
    "code":         r"\w+\([^)\n]*\)|^\s{2,}\S|\b(?:for|if|else|print|return)\b.*:",
    "list":         r"(?m)^\s*(?:\d+[.)]|[A-Z][.)]|[-*•])\s",
    "quote":        r"[.?!][\"”')]",
    "many newlines": r"(?:[^\n]*\n){6}",
}

def pick_preview_rows(n=N_PREVIEW):
    texts = df[TEXT_COL]
    candidates = texts[texts.notna() & texts.astype(str).str.strip().ne("")]
    if len(candidates) <= n:
        return list(candidates.index), {i: [] for i in candidates.index}
    tags = {i: [k for k, p in EDGE_PATTERNS.items() if re.search(p, candidates[i])] for i in candidates.index}
    chosen = []
    shuffled = candidates.sample(frac=1, random_state=SEED).index
    for k in EDGE_PATTERNS:                        # one row per edge-case type first (up to half of n)
        if len(chosen) >= n // 2:
            break
        hit = next((i for i in shuffled if k in tags[i] and i not in chosen), None)
        if hit is not None:
            chosen.append(hit)
    chosen += [i for i in shuffled if i not in chosen][: n - len(chosen)]
    return chosen, tags

def show_rows(indices, tags=None, height=380):
    box = ("white-space:pre-wrap;word-break:break-word;font-family:monospace;font-size:12px;"
           f"max-height:{height}px;overflow:auto;border:1px solid #8885;padding:6px;border-radius:4px")
    mark = "background:rgba(255,190,0,.45);border-radius:3px;padding:0 1px;font-weight:600"
    parts = []
    for i in indices:
        left = html.escape(str(df.at[i, TEXT_COL]))
        right = MARKER_RE.sub(lambda m: f'<span style="{mark}">{m.group()}</span>', html.escape(str(out.at[i, TEXT_COL])))
        n_mark = len(MARKER_RE.findall(str(out.at[i, TEXT_COL])))
        tag = f" · edge cases: {', '.join(tags[i])}" if tags and tags.get(i) else ""
        parts.append(f"<div style='margin:14px 0 4px;font-weight:600'>row index {i} · {n_mark} sentences{tag}</div>"
                     f"<div style='display:grid;grid-template-columns:1fr 1fr;gap:8px'>"
                     f"<div style='{box}'>{left}</div><div style='{box}'>{right}</div></div>")
    display(HTML("".join(parts)))

preview_idx, preview_tags = pick_preview_rows()
show_rows(preview_idx, preview_tags)

To look at specific rows, call `show_rows([12, 40])`. If something looks wrong, adjust the rules in section 3 and re-run from there.

## 6. Save results

In [12]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
ext = out_path.suffix.lower()
if ext == ".parquet":
    out.to_parquet(out_path, index=False)          # needs pyarrow or fastparquet
elif ext in (".csv", ".tsv"):
    out.to_csv(out_path, index=False, sep="\t" if ext == ".tsv" else ",")
else:
    raise ValueError(f"Unsupported output format '{ext}' (use .csv, .tsv or .parquet)")

# re-load what was written and re-check the round trip
check_df = load_table(out_path) if ext != ".tsv" else pd.read_csv(out_path, sep="\t", **{k: v for k, v in CSV_KWARGS.items() if k != "sep"})
assert check_df.shape == out.shape and list(check_df.columns) == list(out.columns), "saved file differs in shape/columns"
for a, b in zip(df[TEXT_COL], check_df[TEXT_COL]):
    if isinstance(a, str) and a.strip():
        assert strip_markers(b) == a, "saved text does not match original"

print(f"Saved: {out_path.resolve()}")

Saved: C:\Users\afath\Desktop\Orange Lab\test-time-compute-allocation\src\data\processed\LiquidAILFM2.5-2.6B\LiquidAILFM2.5-2.6B_raw_results_segmented.csv
